## Extraction Pipeline

This pipeline is responsible to extract CSV files from the RAW directory and load the files as PARQUET in the BRONZE directory.

All columns are parsed as str.

Only minor changes to column titles are made.

In [0]:
import os
from pyspark.sql.types import StructType, StructField, StringType
from typing import Optional
from pyspark.sql import DataFrame

raw_volume_dir = "/Volumes/workspace/default/garabage_collection_volume/raw/"
bronze_volume_dir = "/Volumes/workspace/default/garabage_collection_volume/bronze/"

In [0]:
def stop_pipeline(msg: str):
    dbutils.jobs.taskValues.set("STOP", "1")
    dbutils.notebook.exit(msg)
    raise RuntimeError(msg)

In [0]:
def get_unparsed_file() -> Optional[str]:
    raw_file_list = [raw_file.split('.')[0] for raw_file in os.listdir(raw_volume_dir)]
    bronze_file_list = [bronze_file.split('.')[0] for bronze_file in os.listdir(bronze_volume_dir)]
    for raw_file in raw_file_list:
        if raw_file not in bronze_file_list:
            return raw_file

In [0]:
schema = StructType([
    StructField('ANO', StringType(), True),
    StructField('MÒS', StringType(), True),
    StructField('REGIONAL', StringType(), True),
    StructField('MODALIDADE', StringType(), True),
    StructField('OPERADOR', StringType(), True),
    StructField('DISTRITO', StringType(), True),
    StructField('MASSA (t)', StringType(), True),
    StructField('VIAGENS', StringType(), True),
    StructField('PERCURSO (Km)', StringType(), True),
    StructField('Hora (h:mm)', StringType(), True),
    StructField('DIAS', StringType(), True),
    StructField('COLETORES', StringType(), True),
    StructField('HORAS', StringType(), True),
    StructField('CLASSE', StringType(), True),
])
columns_to_rename = {
    'MÒS': 'MES',
    'MASSA (t)': 'MASSA',
    'PERCURSO (Km)': 'PERCURSO',
    'HORA (h:mm)': 'HORA'
}

In [0]:
def extract(file_name: str) -> DataFrame:
    file_to_parse = os.path.join(raw_volume_dir, f"{file_name}.csv")
    df_file_to_parse = spark.read.format('csv').option('header','true').schema(schema).load(file_to_parse)
    df_file_to_parse = df_file_to_parse.withColumnsRenamed(columns_to_rename)
    return df_file_to_parse

In [0]:
def load(df: DataFrame, file_name: str):
    file_to_parse = os.path.join(bronze_volume_dir, f"{file_name}.parquet")
    df.write.format('parquet').mode('overwrite').save(file_to_parse)

In [0]:
# Get file that was not parsed
file_name = get_unparsed_file()
if not file_name:
    stop_pipeline("Not found any new files.")
file_name

# Run pipeline
df = extract(file_name)
load(df, file_name)

# Save file name for next job
dbutils.jobs.taskValues.set(key="file_name", value=file_name)